# Hand localization and validation protocol: experiments A, B, C
**Nurzhan Nurlybek, Astana IT University**

All numbers in the paper come from this notebook.

| Experiment | Question | Cost |
|---|---|---|
| **A** | How does accuracy depend on how the hand is localized? Same model, same 500 unseen photos, only the crop changes | inference only |
| **B** | How do the numbers change with the validation protocol: one small test split, random 5-fold, group 5-fold? | ~14 trainings per seed |
| **C** | Does the model transfer between HaGRID photos and own photos (both ways)? Does the low-resolution augmentation help? | ~13 trainings per seed |

**How to run:** Runtime → Change runtime type → **T4 GPU**. Put `dataset.zip` (the 500 images and the metadata files `hagrid_meta.csv` and `own_meta_final.csv`) into `MyDrive/CV_Assignment1/`.
1. **Quick check first:** in the setup cell set `SMOKE = True` and run all cells (about 10 minutes, mostly downloads). If it ends without errors, set `SMOKE = False`.
2. **Full run:** run all cells in order. Everything is saved to `MyDrive/CV_Assignment1/experiments/` after each step, so if Colab disconnects, just run the cells again from the top: finished steps are skipped.

Seeds run one after another (seed 0 first). After seed 0 you can already run the analysis cells at the bottom.

## 1. Setup

In [ ]:
import os, sys, time, json, glob, random, shutil, zipfile
IN_COLAB = 'google.colab' in sys.modules
DRY_RUN = os.environ.get('DRY_RUN') == '1'   # offline self-test only; never set in Colab
SMOKE = False   # True = 10-minute check of the whole pipeline (tiny training, 1 seed, results in experiments_smoke/)

if IN_COLAB:
    os.system('pip install -q mediapipe==0.10.32')
    if not os.path.exists('hand_landmarker.task'):
        os.system('wget -q -nc https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task')
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE = os.environ.get('EXP_DRIVE', '/content/drive/MyDrive/CV_Assignment1')
HAGRID = os.environ.get('EXP_HAGRID', '/content/hagrid')
DATASET = os.environ.get('EXP_DATASET', '/content/dataset')
OUT = os.environ.get('EXP_OUT', f'{DRIVE}/experiments' + ('_smoke' if SMOKE else ''))
for d in (OUT, f'{OUT}/tables', f'{OUT}/figures'):
    os.makedirs(d, exist_ok=True)

if IN_COLAB and not os.path.isdir(HAGRID):   # public HaGRID sample (384 px), no Kaggle account needed
    import kagglehub
    os.symlink(kagglehub.dataset_download('innominate817/hagrid-sample-30k-384p'), HAGRID)
    assert os.path.isdir(HAGRID), 'download of the HaGRID sample failed: rerun this cell'
if not os.path.exists(f'{DATASET}/hagrid_meta.csv'):
    os.makedirs(DATASET, exist_ok=True)
    shutil.unpack_archive(f'{DRIVE}/dataset.zip', DATASET)
assert os.path.exists(f'{DATASET}/hagrid_meta.csv') and os.path.exists(f'{DATASET}/own_meta_final.csv'), \
    'dataset.zip must contain hagrid_meta.csv and own_meta_final.csv'
print('setup ok | dry run:', DRY_RUN)

In [ ]:
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from skimage.feature import hog

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DRY_RUN or device == 'cuda', 'Turn on the GPU: Runtime -> Change runtime type -> T4 GPU'

CLASSES = ['fist', 'stop', 'like', 'peace', 'ok']
QUICK = DRY_RUN or SMOKE
SEEDS = [0] if QUICK else [0, 1, 2]          # edit to [0] to run one seed only
EPOCHS = (1, 1) if QUICK else (5, 15)        # stage 1 / stage 2, as in Assignment 1
PRETRAINED = not DRY_RUN
NUM_WORKERS = 2 if IN_COLAB else 0
HOLDOUT_PER_SEED = 2 if DRY_RUN else (1 if SMOKE else 4)   # small-test-split repeats per seed (experiment B)
N_RAW = 8 if DRY_RUN else (20 if SMOKE else 100)           # raw photos per class (experiment A)
IMG_SIZE = 64 if DRY_RUN else 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
PAD_COLOR = (124, 116, 104)                  # ImageNet mean, used for letterbox padding
JITTER = [0.1, 0.2, 0.3, 0.5]                # box noise levels (experiment A)
T0 = time.time()

def log(msg):
    print(f'[{(time.time() - T0) / 60:6.1f} min] {msg}', flush=True)

HAS_MP = os.path.exists('hand_landmarker.task')
if HAS_MP:
    import mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision
assert HAS_MP or DRY_RUN, 'hand_landmarker.task is missing: rerun the setup cell'
print('device:', device, '| seeds:', SEEDS, '| epochs:', EPOCHS)

## 2. Helpers (copied from Assignment 1 where possible)

In [ ]:
def square_region(W, H, x0, y0, x1, y1, margin):
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    side = max(x1 - x0, y1 - y0) * (1 + margin)
    return (max(0, cx - side / 2), max(0, cy - side / 2), min(W, cx + side / 2), min(H, cy + side / 2))

def region_iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

if HAS_MP:
    def make_detector(conf):
        return vision.HandLandmarker.create_from_options(
            vision.HandLandmarkerOptions(
                base_options=mp_python.BaseOptions(model_asset_path='hand_landmarker.task'),
                num_hands=1,
                min_hand_detection_confidence=conf,
                min_hand_presence_confidence=conf))

    def locate_hand_box(img, detectors):
        # same search as locate_hand() in Assignment 1, but returns the landmark box instead of a crop
        for name, det in detectors:
            for size in (None, 640):
                im = img.copy()
                if size:
                    im.thumbnail((size, size))
                for flip in (False, True):
                    src = ImageOps.mirror(im) if flip else im
                    res = det.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.array(src)))
                    if res.hand_landmarks:
                        lm = res.hand_landmarks[0]
                        xs = [(1 - p.x) if flip else p.x for p in lm]
                        ys = [p.y for p in lm]
                        W, H = img.size
                        return (min(xs) * W, min(ys) * H, max(xs) * W, max(ys) * H), name
        return None, 'full frame'
else:
    # offline self-test only: a fake detector that "finds" a box around the centre most of the time
    def make_detector(conf):
        return None

    def locate_hand_box(img, detectors):
        W, H = img.size
        k = (W * 31 + H * 17 + int(np.array(img)[0, 0].sum())) % 10
        if k < 2:
            return None, 'full frame'
        return (W * 0.3, H * 0.3, W * 0.7, H * 0.7), 'hand landmarks'


class LowResolution:
    def __init__(self, p=0.3):
        self.p = p
    def __call__(self, img):
        if random.random() < self.p:
            w, h = img.size
            s = random.uniform(0.25, 0.6)
            img = img.resize((max(1, int(w * s)), max(1, int(h * s)))).resize((w, h))
        return img

def make_train_tf(lowres_p=0.3):
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        LowResolution(lowres_p),
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
        transforms.RandomApply([transforms.GaussianBlur(5)], p=0.2),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD)])

EVAL_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)])

IMG = {}   # path -> PIL image, filled in section 3 (keeps the images in RAM, no disk reads while training)

class GestureDataset(Dataset):
    def __init__(self, frame, tf):
        self.paths, self.ys, self.tf = frame['path'].tolist(), frame['y'].tolist(), tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.tf(IMG[self.paths[i]]), self.ys[i]

def run_epoch(net, loader, crit, optimizer=None):
    training = optimizer is not None
    net.train(training)
    total, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = net(x)
            loss = crit(out, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            n += len(y)
    return total / n, correct / n

def fit_mobilenet(tr, va, seed, lowres_p=0.3):
    # same recipe as Assignment 1: stage 1 head only (lr 1e-3), stage 2 last 5 blocks (lr 1e-4), best val epoch kept
    t = time.time()
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    weights = models.MobileNet_V2_Weights.IMAGENET1K_V1 if PRETRAINED else None
    net = models.mobilenet_v2(weights=weights)
    net.classifier[1] = nn.Linear(net.last_channel, len(CLASSES))
    net = net.to(device)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    tl = DataLoader(GestureDataset(tr, make_train_tf(lowres_p)), batch_size=32, shuffle=True, num_workers=NUM_WORKERS)
    vl = DataLoader(GestureDataset(va, EVAL_TF), batch_size=32, num_workers=0)
    best, best_state = -1.0, None
    for stage, (epochs, lr) in enumerate(zip(EPOCHS, (1e-3, 1e-4))):
        for p in net.features.parameters():
            p.requires_grad = False
        if stage == 1:
            for p in net.features[-5:].parameters():
                p.requires_grad = True
        opt = torch.optim.Adam([p for p in net.parameters() if p.requires_grad], lr=lr)
        for _ in range(epochs):
            run_epoch(net, tl, crit, opt)
            _, acc = run_epoch(net, vl, crit)
            if acc >= best:
                best = acc
                best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
    net.load_state_dict(best_state)
    log(f'trained on {len(tr)} (val {len(va)}) in {time.time() - t:.0f}s, best val acc {best:.3f}')
    return net.eval()

def predict_frame(net, frame):
    loader = DataLoader(GestureDataset(frame, EVAL_TF), batch_size=64, num_workers=0)
    with torch.no_grad():
        return np.concatenate([net(x.to(device)).argmax(1).cpu().numpy() for x, _ in loader])

def predict_pil(net, imgs, bs=64):
    probs = []
    with torch.no_grad():
        for i in range(0, len(imgs), bs):
            x = torch.stack([EVAL_TF(im) for im in imgs[i:i + bs]]).to(device)
            probs.append(torch.softmax(net(x), 1).cpu().numpy())
    return np.concatenate(probs)

def svm_predict(tr, te):
    clf = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=10, gamma='scale')).fit(X_HOG[tr.index], tr['y'])
    return clf.predict(X_HOG[te.index])

def group_val_split(frame, seed):
    a, b = next(StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=seed).split(frame, frame['y'], frame['group']))
    return frame.iloc[a], frame.iloc[b]

def random_val_split(frame, seed):
    a, b = next(StratifiedKFold(n_splits=6, shuffle=True, random_state=seed).split(frame, frame['y']))
    return frame.iloc[a], frame.iloc[b]

def cached(path, fn):
    if os.path.exists(path):
        log(f'skip (already done): {os.path.basename(path)}')
        return pd.read_csv(path)
    out = fn()
    out.to_csv(path, index=False)
    log(f'saved {os.path.basename(path)} ({len(out)} rows)')
    return out

## 3. Data (the 500 images of Assignment 1)

In [ ]:
hagrid = pd.read_csv(f'{DATASET}/hagrid_meta.csv')
own = pd.read_csv(f'{DATASET}/own_meta_final.csv')
df = pd.concat([hagrid, own], ignore_index=True)
df['y'] = df['label'].map({c: i for i, c in enumerate(CLASSES)})
print(df.groupby(['source', 'label']).size().unstack())
print('groups:', df.groupby('source')['group'].nunique().to_dict())
if not DRY_RUN:
    assert len(df) == 500 and df.groupby(['source', 'label']).size().eq(50).all(), 'unexpected dataset size'

def fix_path(p, label):
    if os.path.exists(p):
        return p
    alt = f'{DATASET}/{label}/{os.path.basename(p)}'
    assert os.path.exists(alt), f'image not found: {p}'
    return alt

df['path'] = [fix_path(p, l) for p, l in zip(df['path'], df['label'])]
IMG.update({p: Image.open(p).convert('RGB') for p in df['path']})

def hog_vec(img):
    return hog(np.array(img.convert('L').resize((128, 128))), orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2))

X_HOG = np.array([hog_vec(IMG[p]) for p in df['path']])
print('HOG features:', X_HOG.shape)

## 4. Experiments B and C (retraining)
**B – validation protocol.** The same 500 images, the same models, three ways to measure accuracy:
`random` 5-fold (photos of the same person or situation can be in train and test), `group` 5-fold (the protocol of Assignment 1), and `holdout` (one small group-aware split of about 70 images, repeated with different random splits).
`group_in_train` records, for every test image, whether another photo of the same person or situation was in the training part of the fold.

**C – domain transfer.** Train on HaGRID photos only and test on own photos (and the other way round); compare with group 5-fold inside each source. Extra run: HaGRID → own without the low-resolution augmentation.

In [ ]:
def result_rows(te, **kw):
    out = pd.DataFrame({'idx': te.index.values, 'source': te['source'].values, 'y': te['y'].values})
    for k, v in kw.items():
        out[k] = v
    return out

def run_B(seed):
    parts = []
    for proto in ('random', 'group'):
        if proto == 'random':
            splits = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed).split(df, df['y'])
        else:
            splits = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed).split(df, df['y'], df['group'])
        for k, (trv_i, te_i) in enumerate(splits, 1):
            trv, te = df.iloc[trv_i], df.iloc[te_i]
            tr, va = (random_val_split if proto == 'random' else group_val_split)(trv, seed)
            in_train = te['group'].isin(set(trv['group'])).values
            if proto == 'group':
                assert not in_train.any(), 'group protocol leaked'
            log(f'B {proto} seed {seed} fold {k}')
            parts.append(result_rows(te, exp='B', protocol=proto, seed=seed, fold=k,
                                     svm=svm_predict(tr, te), cnn=predict_frame(fit_mobilenet(tr, va, seed), te),
                                     group_in_train=in_train))
    for j in range(HOLDOUT_PER_SEED):
        rs = seed * 100 + j
        trv_i, te_i = next(StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=rs).split(df, df['y'], df['group']))
        trv, te = df.iloc[trv_i], df.iloc[te_i]
        tr, va = group_val_split(trv, rs)
        log(f'B holdout seed {seed} split {j}')
        parts.append(result_rows(te, exp='B', protocol='holdout', seed=seed, fold=j,
                                 svm=svm_predict(tr, te), cnn=predict_frame(fit_mobilenet(tr, va, rs), te),
                                 group_in_train=False))
    return pd.concat(parts, ignore_index=True)

def run_C(seed):
    parts = []
    src = {'hagrid': df[df.source == 'hagrid'], 'own': df[df.source == 'own']}
    for a, b in (('hagrid', 'own'), ('own', 'hagrid')):
        tr, va = group_val_split(src[a], seed)
        te = src[b]
        log(f'C {a}->{b} seed {seed}')
        parts.append(result_rows(te, exp='C', setting=f'{a}->{b}', seed=seed, fold=0,
                                 svm=svm_predict(tr, te), cnn=predict_frame(fit_mobilenet(tr, va, seed), te)))
    tr, va = group_val_split(src['hagrid'], seed)
    te = src['own']
    log(f'C hagrid->own without low-res augmentation, seed {seed}')
    parts.append(result_rows(te, exp='C', setting='hagrid->own, no low-res aug', seed=seed, fold=0,
                             svm=svm_predict(tr, te), cnn=predict_frame(fit_mobilenet(tr, va, seed, lowres_p=0.0), te)))
    for name in ('hagrid', 'own'):
        s = src[name]
        for k, (trv_i, te_i) in enumerate(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed).split(s, s['y'], s['group']), 1):
            trv, te = s.iloc[trv_i], s.iloc[te_i]
            tr, va = group_val_split(trv, seed)
            log(f'C in-domain {name} seed {seed} fold {k}')
            parts.append(result_rows(te, exp='C', setting=f'{name} in-domain CV', seed=seed, fold=k,
                                     svm=svm_predict(tr, te), cnn=predict_frame(fit_mobilenet(tr, va, seed), te)))
    return pd.concat(parts, ignore_index=True)

## 5. Experiment A: how the hand is localized
The 500 raw HaGRID photos of Assignment 1 are rebuilt with the same seeds (`random.Random(123)`); their people do not appear in the 500-image dataset. The classifier is trained on the 500 images (group-aware validation split) and then sees the **same** photos cropped in different ways:

| Variant | Crop |
|---|---|
| `gt_m0.00`, `gt_m0.25`, `gt_m0.50`, `gt_m1.00` | box from the HaGRID annotation, margin 0 / 25 / 50 / 100 % |
| `jit_0.1` … `jit_0.5` | annotation box with random shift and scale error up to 10 / 20 / 30 / 50 %, margin 25 % (a detector with controlled error) |
| `mp_m0.25`, `mp_m0.35` | MediaPipe Hand Landmarker box (the pipeline of Assignment 1), margin 25 % / 35 %; whole photo if no hand is found |
| `whole_squash`, `whole_pad` | whole photo, squeezed to a square / padded to a square |

For every crop we also record its overlap (IoU) with the `gt_m0.25` crop, a direct measure of localization quality.

In [ ]:
def build_raw():
    img_paths = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(f'{HAGRID}/**/*.jpg', recursive=True)}
    ann_files = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(f'{HAGRID}/**/*.json', recursive=True)}
    used = set(hagrid['original'])
    seen_people = set(df['group'])
    rng = random.Random(123)
    rows = []
    for cls in CLASSES:
        ann = json.load(open(ann_files[cls]))
        pool = sorted(i for i in ann if i in img_paths and i not in used and cls in ann[i]['labels']
                      and f"hagrid_{ann[i].get('user_id', i)}" not in seen_people)
        for i in rng.sample(pool, N_RAW):
            a = ann[i]
            x, y, w, h = a['bboxes'][a['labels'].index(cls)]
            rows.append({'id': i, 'label': cls, 'y': CLASSES.index(cls), 'path': img_paths[i],
                         'person': f"hagrid_{a.get('user_id', i)}", 'bx': x, 'by': y, 'bw': w, 'bh': h})
    return pd.DataFrame(rows)

raw = build_raw()
assert not set(raw['person']) & set(df['group']) and not set(raw['id']) & set(hagrid['original'])
RAW_IMGS = [Image.open(p).convert('RGB') for p in raw['path']]
raw['W'] = [im.size[0] for im in RAW_IMGS]
raw['H'] = [im.size[1] for im in RAW_IMGS]
raw['gx0'], raw['gy0'] = raw['bx'] * raw['W'], raw['by'] * raw['H']
raw['gx1'], raw['gy1'] = (raw['bx'] + raw['bw']) * raw['W'], (raw['by'] + raw['bh']) * raw['H']
raw.drop(columns=['path']).to_csv(f'{OUT}/A_raw_photos.csv', index=False)
print(len(raw), 'raw photos |', raw['person'].nunique(), 'people | per class:', raw['label'].value_counts().to_dict())

def compute_mp():
    dets = [('hand landmarks', make_detector(0.3)), ('hand landmarks, relaxed', make_detector(0.1))]
    rows = []
    for k, im in enumerate(RAW_IMGS):
        box, method = locate_hand_box(im, dets)
        rows.append({'id': raw['id'][k], 'found': box is not None, 'method': method,
                     'mx0': box[0] if box else np.nan, 'my0': box[1] if box else np.nan,
                     'mx1': box[2] if box else np.nan, 'my1': box[3] if box else np.nan})
    for _, d in dets:
        if d is not None:
            d.close()
    return pd.DataFrame(rows)

mpb = cached(f'{OUT}/A_mediapipe_boxes.csv', compute_mp)
raw = raw.merge(mpb, on='id', how='left')
print('MediaPipe found the hand in', int(raw['found'].sum()), 'of', len(raw), 'photos')

A_VARIANTS = (['gt_m0.00', 'gt_m0.25', 'gt_m0.50', 'gt_m1.00'] + [f'jit_{L:.1f}' for L in JITTER]
              + ['mp_m0.25', 'mp_m0.35', 'whole_squash', 'whole_pad'])

def build_variants(k, seed):
    r = raw.iloc[k]
    img, W, H = RAW_IMGS[k], int(r['W']), int(r['H'])
    g = (r['gx0'], r['gy0'], r['gx1'], r['gy1'])
    out = {}
    for m in (0.0, 0.25, 0.5, 1.0):
        reg = square_region(W, H, *g, m)
        out[f'gt_m{m:.2f}'] = (img.crop(reg), reg)
    gw, gh = g[2] - g[0], g[3] - g[1]
    for li, L in enumerate(JITTER):
        rng = np.random.default_rng([seed, li, k])
        cx = (g[0] + g[2]) / 2 + rng.uniform(-L, L) * gw
        cy = (g[1] + g[3]) / 2 + rng.uniform(-L, L) * gh
        w, h = gw * (1 + rng.uniform(-L, L)), gh * (1 + rng.uniform(-L, L))
        reg = square_region(W, H, cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2, 0.25)
        out[f'jit_{L:.1f}'] = (img.crop(reg), reg)
    full = (0.0, 0.0, float(W), float(H))
    for m in (0.25, 0.35):
        if bool(r['found']):
            reg = square_region(W, H, r['mx0'], r['my0'], r['mx1'], r['my1'], m)
            out[f'mp_m{m:.2f}'] = (img.crop(reg), reg)
        else:
            out[f'mp_m{m:.2f}'] = (img, full)
    out['whole_squash'] = (img, full)
    s = max(W, H)
    out['whole_pad'] = (ImageOps.pad(img, (s, s), color=PAD_COLOR), full)
    return out

def run_A(seed):
    tr, va = group_val_split(df, seed)
    net = fit_mobilenet(tr, va, seed)
    per_photo = [build_variants(k, seed) for k in range(len(raw))]
    parts = []
    for v in A_VARIANTS:
        probs = predict_pil(net, [pp[v][0] for pp in per_photo])
        ious = [region_iou(pp[v][1], pp['gt_m0.25'][1]) for pp in per_photo]
        parts.append(pd.DataFrame({'exp': 'A', 'seed': seed, 'variant': v, 'id': raw['id'].values,
                                   'person': raw['person'].values, 'label': raw['label'].values, 'y': raw['y'].values,
                                   'found': raw['found'].values, 'pred': probs.argmax(1), 'conf': probs.max(1),
                                   'crop_iou': ious}))
    return pd.concat(parts, ignore_index=True)

## 6. Run everything
Seed 0 first (about 30 trainings). Finished steps are saved and skipped on rerun. Interrupt (stop the cell) whenever you want: the analysis cells below work with whatever seeds are finished.

In [ ]:
for seed in SEEDS:
    log(f'================ seed {seed} ================')
    cached(f'{OUT}/A_seed{seed}.csv', lambda: run_A(seed))
    cached(f'{OUT}/B_seed{seed}.csv', lambda: run_B(seed))
    cached(f'{OUT}/C_seed{seed}.csv', lambda: run_C(seed))
log('all requested seeds are finished')

## 7. Analysis A: localization

In [ ]:
plt.rcParams.update({'font.size': 8, 'font.family': 'serif',
                     'font.serif': ['Times New Roman', 'Liberation Serif', 'Nimbus Roman', 'DejaVu Serif'],
                     'axes.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False})

VARIANT_NAMES = {
    'gt_m0.00': 'Annotated box, margin 0%', 'gt_m0.25': 'Annotated box, margin 25%',
    'gt_m0.50': 'Annotated box, margin 50%', 'gt_m1.00': 'Annotated box, margin 100%',
    'jit_0.1': 'Noisy box, error 10%', 'jit_0.2': 'Noisy box, error 20%',
    'jit_0.3': 'Noisy box, error 30%', 'jit_0.5': 'Noisy box, error 50%',
    'mp_m0.25': 'MediaPipe, margin 25%', 'mp_m0.35': 'MediaPipe, margin 35%',
    'whole_squash': 'Whole photo, squeezed', 'whole_pad': 'Whole photo, padded'}

def load_all(prefix):
    files = sorted(glob.glob(f'{OUT}/{prefix}_seed*.csv'))
    assert files, f'no {prefix}_seed*.csv yet: run section 6 first'
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

def cboot_ci(values, clusters, n_boot=2000, seed=0):
    # 95% bootstrap interval of the mean, resampling whole clusters (people), not single photos
    values = np.asarray(values, float)
    codes, uniq = pd.factorize(pd.Series(np.asarray(clusters)))
    sums = np.bincount(codes, weights=values, minlength=len(uniq))
    cnts = np.bincount(codes, minlength=len(uniq))
    idx = np.random.default_rng(seed).integers(0, len(uniq), size=(n_boot, len(uniq)))
    stat = sums[idx].sum(1) / cnts[idx].sum(1)
    return np.percentile(stat, [2.5, 97.5])

def save_fig(fig, name):
    fig.savefig(f'{OUT}/figures/{name}.png', dpi=600, bbox_inches='tight')
    fig.savefig(f'{OUT}/figures/{name}.jpg', dpi=600, bbox_inches='tight', pil_kwargs={'quality': 95})

def save_table(t, name):
    t.to_csv(f'{OUT}/tables/{name}.csv', index=False)
    print(f'--- {name} ---')
    print(t.to_string(index=False))

A = load_all('A')
A['ok'] = (A['pred'] == A['y']).astype(float)
A['found'] = A['found'].astype(bool)
photo = A.groupby(['variant', 'id', 'person', 'label', 'found'], as_index=False).agg(ok=('ok', 'mean'), iou=('crop_iou', 'mean'))
seed_acc = A.groupby(['variant', 'seed'])['ok'].mean().groupby('variant').agg(['mean', 'std'])
print('seeds used:', [int(x) for x in sorted(A['seed'].unique())])

rows = []
for v in A_VARIANTS:
    g = photo[photo['variant'] == v]
    lo, hi = cboot_ci(g['ok'], g['person'])
    rows.append({'variant': v, 'accuracy_%': 100 * g['ok'].mean(), 'ci95_low': 100 * lo, 'ci95_high': 100 * hi,
                 'seed_std_pp': 100 * seed_acc.loc[v, 'std'] if not np.isnan(seed_acc.loc[v, 'std']) else np.nan,
                 'mean_crop_iou': g['iou'].mean(), 'n_photos': len(g)})
tabA1 = pd.DataFrame(rows).round(2)
save_table(tabA1, 'A1_accuracy_by_localization_variant')

wide = photo.pivot_table(index=['id', 'person', 'found'], columns='variant', values='ok').reset_index()
rows = []
for name, sub in (('MediaPipe found the hand', wide[wide['found']]), ('MediaPipe did not find the hand', wide[~wide['found']]),
                  ('all photos', wide)):
    for a, b in (('gt_m0.25', 'mp_m0.25'), ('gt_m0.25', 'mp_m0.35'), ('gt_m0.25', 'whole_squash'), ('gt_m0.25', 'whole_pad')):
        if len(sub) < 5:
            continue
        d = (sub[a] - sub[b]).values
        lo, hi = cboot_ci(d, sub['person'])
        rows.append({'photos': name, 'n': len(sub), 'reference': a, 'alternative': b, 'reference_%': 100 * sub[a].mean(),
                     'alternative_%': 100 * sub[b].mean(), 'diff_pp': 100 * d.mean(), 'ci95_low_pp': 100 * lo, 'ci95_high_pp': 100 * hi})
tabA2 = pd.DataFrame(rows).round(2)
save_table(tabA2, 'A2_same_photos_paired_differences')

fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.8), gridspec_kw={'width_ratios': [1, 1.05]})
crops = A[A['variant'].str.startswith(('gt_', 'jit_')) | (A['variant'].str.startswith('mp_') & A['found'])]
edges = np.array([0, 0.2, 0.4, 0.6, 0.8, 1.0001])
xs, ys, los, his = [], [], [], []
for lo_e, hi_e in zip(edges[:-1], edges[1:]):
    part = crops[(crops['crop_iou'] >= lo_e) & (crops['crop_iou'] < hi_e)]
    if len(part) < 20:
        continue
    lo, hi = cboot_ci(part['ok'], part['person'])
    xs.append((lo_e + min(hi_e, 1.0)) / 2); ys.append(100 * part['ok'].mean()); los.append(100 * lo); his.append(100 * hi)
axes[0].errorbar(xs, ys, yerr=[np.array(ys) - np.array(los), np.array(his) - np.array(ys)], marker='o', ms=3, lw=1, capsize=2, color='black')
axes[0].set_xlabel('Overlap with the annotated crop (intersection over union)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('(a)', loc='left', fontsize=8)
axes[0].set_ylim(0, 100)
t = tabA1.iloc[::-1]
axes[1].barh(t['variant'].map(VARIANT_NAMES), t['accuracy_%'], xerr=[t['accuracy_%'] - t['ci95_low'], t['ci95_high'] - t['accuracy_%']],
             color='0.7', ecolor='black', capsize=2, height=0.7)
axes[1].set_xlabel('Accuracy (%)')
axes[1].set_xlim(0, 100)
axes[1].set_title('(b)', loc='left', fontsize=8)
axes[1].tick_params(axis='y', labelsize=7)
plt.tight_layout()
save_fig(fig, 'fig_A_localization')
plt.show()

In [ ]:
# Figure 1 of the paper: one raw photo, all localization variants (hands only; the whole-photo variants show the context)
k0 = int(raw.index[(raw['label'] == 'peace') & raw['found']][0])
vs = build_variants(k0, 0)
fig, axes = plt.subplots(2, 6, figsize=(6.5, 2.6))
for ax, v in zip(axes.ravel(), A_VARIANTS):
    ax.imshow(vs[v][0]); ax.set_title(VARIANT_NAMES[v].replace(', ', '\n'), fontsize=6); ax.axis('off')
plt.tight_layout()
save_fig(fig, 'fig_1_localization_variants')
plt.show()

## 8. Analysis B: validation protocol

In [ ]:
B = load_all('B')
B['svm_ok'] = (B['svm'] == B['y']).astype(float)
B['cnn_ok'] = (B['cnn'] == B['y']).astype(float)
B['group_in_train'] = B['group_in_train'].astype(bool)
print('seeds used:', [int(x) for x in sorted(B['seed'].unique())])

cv = B[B['protocol'].isin(['random', 'group'])]
per_seed = cv.groupby(['protocol', 'seed']).agg(svm=('svm_ok', 'mean'), cnn=('cnn_ok', 'mean')).reset_index()
rows = []
for proto, g in per_seed.groupby('protocol'):
    rows.append({'protocol': f'{proto} 5-fold CV (all 500 images)', 'n_runs': len(g),
                 'HOG+SVM_%': 100 * g['svm'].mean(), 'HOG+SVM_std_pp': 100 * g['svm'].std(),
                 'MobileNetV2_%': 100 * g['cnn'].mean(), 'MobileNetV2_std_pp': 100 * g['cnn'].std()})
ho = B[B['protocol'] == 'holdout'].groupby(['seed', 'fold']).agg(svm=('svm_ok', 'mean'), cnn=('cnn_ok', 'mean'), n=('y', 'size')).reset_index()
rows.append({'protocol': f'single holdout split (~{int(ho["n"].mean())} test images)', 'n_runs': len(ho),
             'HOG+SVM_%': 100 * ho['svm'].mean(), 'HOG+SVM_std_pp': 100 * ho['svm'].std(),
             'MobileNetV2_%': 100 * ho['cnn'].mean(), 'MobileNetV2_std_pp': 100 * ho['cnn'].std()})
tabB1 = pd.DataFrame(rows).round(2)
save_table(tabB1, 'B1_accuracy_by_validation_protocol')

hmm = pd.DataFrame({'min_%': [100 * ho['cnn'].min(), 100 * ho['svm'].min()], 'max_%': [100 * ho['cnn'].max(), 100 * ho['svm'].max()]},
                   index=['MobileNetV2', 'HOG+SVM']).round(1).reset_index().rename(columns={'index': 'model'})
save_table(hmm, 'B2_holdout_range')

rows = []
for (proto, src), g in cv.groupby(['protocol', 'source']):
    ps = g.groupby('seed').agg(svm=('svm_ok', 'mean'), cnn=('cnn_ok', 'mean'))
    rows.append({'protocol': proto, 'source': src, 'n_images': g['idx'].nunique(),
                 'share_with_same_group_in_train_%': 100 * g['group_in_train'].mean(),
                 'HOG+SVM_%': 100 * ps['svm'].mean(), 'MobileNetV2_%': 100 * ps['cnn'].mean()})
tabB3 = pd.DataFrame(rows).round(2)
save_table(tabB3, 'B3_by_source_and_leakage_diagnostic')

pivot = per_seed.pivot(index='seed', columns='protocol', values='cnn')
if {'random', 'group'} <= set(pivot.columns):
    diff = 100 * (pivot['random'] - pivot['group'])
    print(f'\nMobileNetV2: random minus group 5-fold = {diff.mean():+.2f} pp (per-seed: {diff.round(2).tolist()})')

fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.8), gridspec_kw={'width_ratios': [1, 1.2]})
for j, (col, lab, mk) in enumerate((('cnn', 'MobileNetV2', 'o'), ('svm', 'HOG + SVM', 's'))):
    for i, proto in enumerate(('random', 'group')):
        v = 100 * per_seed[per_seed['protocol'] == proto][col].values
        axes[0].scatter(np.full(len(v), i + (j - 0.5) * 0.25), v, marker=mk, s=14, facecolor='black' if j == 0 else 'white',
                        edgecolor='black', lw=0.7, label=lab if i == 0 else None)
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['Random split', 'Group split'])
axes[0].set_xlim(-0.6, 1.6); axes[0].set_ylabel('Accuracy over 500 images (%)'); axes[0].legend(frameon=False, fontsize=7)
axes[0].set_title('(a) 5-fold cross-validation, one point per seed', loc='left', fontsize=8)
rng = np.random.default_rng(0)
for j, (col, lab, mk) in enumerate((('cnn', 'MobileNetV2', 'o'), ('svm', 'HOG + SVM', 's'))):
    v = 100 * ho[col].values
    axes[1].scatter(j + rng.uniform(-0.12, 0.12, len(v)), v, marker=mk, s=14, facecolor='black' if j == 0 else 'white', edgecolor='black', lw=0.7)
    gm = 100 * per_seed[per_seed['protocol'] == 'group'][col].mean()
    axes[1].hlines(gm, j - 0.3, j + 0.3, color='0.4', lw=1, ls='--')
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(['MobileNetV2', 'HOG + SVM']); axes[1].set_xlim(-0.6, 1.6)
axes[1].set_ylabel('Accuracy on one small test split (%)')
axes[1].set_title('(b) single holdout splits; dashed: group 5-fold mean', loc='left', fontsize=8)
plt.tight_layout()
save_fig(fig, 'fig_B_validation_protocol')
plt.show()

## 9. Analysis C: domain transfer

In [ ]:
C = load_all('C')
C['svm_ok'] = (C['svm'] == C['y']).astype(float)
C['cnn_ok'] = (C['cnn'] == C['y']).astype(float)
print('seeds used:', [int(x) for x in sorted(C['seed'].unique())])

order = ['hagrid in-domain CV', 'hagrid->own', 'hagrid->own, no low-res aug', 'own in-domain CV', 'own->hagrid']
ps = C.groupby(['setting', 'seed']).agg(svm=('svm_ok', 'mean'), cnn=('cnn_ok', 'mean'), n=('y', 'size')).reset_index()
rows = []
for s in order:
    g = ps[ps['setting'] == s]
    rows.append({'setting': s, 'test_images': int(g['n'].iloc[0]) if len(g) else 0, 'n_seeds': len(g),
                 'HOG+SVM_%': 100 * g['svm'].mean(), 'HOG+SVM_std_pp': 100 * g['svm'].std(),
                 'MobileNetV2_%': 100 * g['cnn'].mean(), 'MobileNetV2_std_pp': 100 * g['cnn'].std()})
tabC = pd.DataFrame(rows).round(2)
save_table(tabC, 'C1_domain_transfer')

fig, ax = plt.subplots(figsize=(6.5, 2.6))
x = np.arange(len(order))
for j, (col, lab, c) in enumerate((('cnn', 'MobileNetV2', '0.25'), ('svm', 'HOG + SVM', '0.7'))):
    m = [100 * ps[ps['setting'] == s][col].mean() for s in order]
    e = [100 * ps[ps['setting'] == s][col].std() if (ps['setting'] == s).sum() > 1 else 0 for s in order]
    ax.bar(x + (j - 0.5) * 0.38, m, 0.38, yerr=e, capsize=2, color=c, ecolor='black', label=lab)
ax.set_xticks(x); ax.set_xticklabels(['HaGRID\nin-domain', 'HaGRID to\nown photos', 'HaGRID to own,\nno low-res aug.',
                                      'Own photos\nin-domain', 'Own photos\nto HaGRID'], fontsize=7)
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0, 100); ax.legend(frameon=False, fontsize=7)
plt.tight_layout()
save_fig(fig, 'fig_C_domain_transfer')
plt.show()

## 10. Pack the results
The file `results_bundle.zip` (in `MyDrive/CV_Assignment1/experiments/`) contains the result tables, the per-image predictions of every run and the figures.

In [ ]:
info = {'torch': torch.__version__, 'device': torch.cuda.get_device_name(0) if device == 'cuda' else 'cpu',
        'seeds': SEEDS, 'epochs': EPOCHS, 'minutes_since_start_of_session': round((time.time() - T0) / 60, 1)}
json.dump(info, open(f'{OUT}/run_info.json', 'w'), indent=2)
zpath = f'{OUT}/results_bundle.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f'{OUT}/*.csv') + glob.glob(f'{OUT}/tables/*.csv') + glob.glob(f'{OUT}/figures/*.jpg') + [f'{OUT}/run_info.json']):
        z.write(f, os.path.relpath(f, OUT))
print('written:', zpath, f'({os.path.getsize(zpath) / 1e6:.2f} MB)')